# ML-QEM: H₂ VQE Data Generation

**Step 1 of 3**: Generate (noisy, ideal) Pauli expectation value pairs for training the ML error mitigation model.

This notebook reproduces the data generation from Section C of:
> *Machine Learning for Practical Quantum Error Mitigation*, Liao et al., Nature Machine Intelligence (2024)

### What this does
- Builds the H₂ Hamiltonian in Pauli form (5 terms)
- Constructs a 2-qubit TwoLocal VQE ansatz (8 parameters)
- Samples 2000 random parameter vectors θ ∈ [-π, π]
- For each θ: runs the circuit on both a **noiseless statevector** and a **noisy FakeLima simulator**
- Saves the resulting (noisy, ideal) pairs as `.npy` files for use in Step 2 (training)


In [1]:
import numpy as np
import json
import matplotlib.pyplot as plt

from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeLimaV2

print("All imports successful.")
print(f"  qiskit_aer: ok")
print(f"  FakeLimaV2: ok")

All imports successful.
  qiskit_aer: ok
  FakeLimaV2: ok


## 1. H₂ Hamiltonian

The H₂ Hamiltonian at bond length 0.735 Å in STO-3G basis, mapped via Jordan-Wigner:

$$\hat{H} = c_0 II + c_1 IZ + c_2 ZI + c_3 ZZ + c_4 XX$$

Coefficients from Kandala et al. (2017) — the same source cited in the ML-QEM paper.

In [2]:
# H2 Hamiltonian coefficients (Kandala et al. 2017, bond length 0.735 Å)
H2_PAULIS = [
    ("II", -1.0523732),
    ("IZ",  0.3979374),
    ("ZI", -0.3979374),
    ("ZZ", -0.0112801),
    ("XX",  0.1809312),
]

hamiltonian   = SparsePauliOp.from_list(H2_PAULIS)
pauli_labels  = [p for p, _ in H2_PAULIS]
pauli_coeffs  = [c for _, c in H2_PAULIS]
pauli_ops     = [SparsePauliOp.from_list([(label, 1.0)]) for label, _ in H2_PAULIS]

print("H2 Hamiltonian Pauli terms:")
for label, coeff in H2_PAULIS:
    print(f"  {coeff:+.7f} * {label}")
print(f"\nTotal terms: {len(H2_PAULIS)}")

H2 Hamiltonian Pauli terms:
  -1.0523732 * II
  +0.3979374 * IZ
  -0.3979374 * ZI
  -0.0112801 * ZZ
  +0.1809312 * XX

Total terms: 5


## 2. VQE Ansatz

Hardware-efficient TwoLocal ansatz: RY rotations + CNOT entanglement, 1 repetition, 2 qubits.  
This matches the 8-parameter ansatz shown in Fig. 5(b) of the ML-QEM paper.

In [3]:
# TwoLocal ansatz: RY + CNOT, 3 reps, 2 qubits = 8 parameters
ansatz = TwoLocal(
    num_qubits=2,
    rotation_blocks="ry",
    entanglement_blocks="cx",
    entanglement="linear",
    reps=3
)

num_params = ansatz.num_parameters
print(f"Number of parameters: {num_params}")
print()
print(ansatz.decompose())

Number of parameters: 8

     ┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
q_0: ┤ Ry(θ[0]) ├──■──┤ Ry(θ[2]) ├──■──┤ Ry(θ[4]) ├──■──┤ Ry(θ[6]) ├
     ├──────────┤┌─┴─┐├──────────┤┌─┴─┐├──────────┤┌─┴─┐├──────────┤
q_1: ┤ Ry(θ[1]) ├┤ X ├┤ Ry(θ[3]) ├┤ X ├┤ Ry(θ[5]) ├┤ X ├┤ Ry(θ[7]) ├
     └──────────┘└───┘└──────────┘└───┘└──────────┘└───┘└──────────┘


## 3. Noise Model

Using FakeLimaV2 — a simulated backend calibrated from IBM's real Lima device.

Key noise parameters (from ML-QEM paper Table I):
- T1 = 61 µs, T2 = 73 µs  
- Readout error = 3.4 × 10⁻²  
- CNOT error = 1.2 × 10⁻²

In [4]:
# Build noise model from FakeLimaV2
backend     = FakeLimaV2()
noise_model = NoiseModel.from_backend(backend)

print(f"Backend: {backend.name}")
print(f"Basis gates: {noise_model.basis_gates}")

Backend: fake_lima
Basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


## 4. Estimators

- **Ideal estimator**: statevector simulation (exact, no shots, no noise) → training labels  
- **Noisy estimator**: shot-based simulation with FakeLima noise → model inputs  

10,000 shots per expectation value, matching the paper (Appendix C).

In [5]:
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import Estimator as AerEstimatorV1
from qiskit.primitives import StatevectorEstimator

# Extract noise model from backend
noise_model = NoiseModel.from_backend(backend)
print("Noise model basis gates:", noise_model.basis_gates)

# Ideal: exact statevector simulation, no shots
ideal_estimator = StatevectorEstimator()

# Noisy: AerEstimator V1 — only V1 correctly applies noise_model via set_options
noisy_estimator = AerEstimatorV1()
noisy_estimator.set_options(noise_model=noise_model, shots=10000)

print("Estimators ready.")

Noise model basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']
Estimators ready.


## 5. Generate Training Data

Sample 2000 random parameter vectors θ ∈ [-π, π] and collect (noisy, ideal) pairs.  


In [6]:
from qiskit import transpile

# Decompose to basis gates WITHOUT mapping to device qubits
transpiled_ansatz = transpile(
    ansatz,
    basis_gates=['rz', 'sx', 'cx', 'x'],
    optimization_level=1
)

print(f"Num qubits: {transpiled_ansatz.num_qubits}")   # should still be 2
print(f"Depth: {transpiled_ansatz.depth()}")
print(f"Gates: {set(transpiled_ansatz.count_ops().keys())}")

# ── Data generation ──────────────────────────────────────────────────────────
N_SAMPLES = 2000
rng = np.random.default_rng(42)
theta_samples = rng.uniform(-np.pi, np.pi, size=(N_SAMPLES, ansatz.num_parameters))

ideal_data = np.zeros((N_SAMPLES, len(H2_PAULIS)))
noisy_data = np.zeros((N_SAMPLES, len(H2_PAULIS)))

for i, theta in enumerate(theta_samples):
    bound_circuit = transpiled_ansatz.assign_parameters(theta)

    # Ideal: V2 API — list of (circuit, observable) tuples
    ideal_job = ideal_estimator.run([(bound_circuit, op) for op in pauli_ops])
    ideal_results = ideal_job.result()

    # Noisy: V1 API — separate lists for circuits and observables
    noisy_job = noisy_estimator.run(
        [bound_circuit] * len(pauli_ops),
        pauli_ops
    )
    noisy_results = noisy_job.result()

    for j in range(len(H2_PAULIS)):
        ideal_data[i, j] = ideal_results[j].data.evs
        noisy_data[i, j] = noisy_results.values[j]

    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{N_SAMPLES} samples done")

print("Done.")

Num qubits: 2
Depth: 19
Gates: {'rz', 'sx', 'cx'}
  200/2000 samples done
  400/2000 samples done
  600/2000 samples done
  800/2000 samples done
  1000/2000 samples done
  1200/2000 samples done
  1400/2000 samples done
  1600/2000 samples done
  1800/2000 samples done
  2000/2000 samples done
Done.


## 6. Save Dataset

In [ ]:
np.save("ideal_data.npy", ideal_data)
np.save("noisy_data.npy", noisy_data)
np.save("theta_samples.npy", theta_samples)

meta = {
    "pauli_labels":       pauli_labels,
    "hamiltonian_coeffs": pauli_coeffs,
    "n_samples":          N_SAMPLES,
    "n_params":           num_params,
    "shots":              10000,
    "noise_backend":      "FakeLimaV2",
    "param_range":        [-np.pi, np.pi],
}
with open("dataset_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved files:")
print("  ideal_data.npy      — ideal (noiseless) expectation values, shape (2000, 5)")
print("  noisy_data.npy      — noisy expectation values,             shape (2000, 5)")
print("  theta_samples.npy   — parameter vectors,                    shape (2000, 8)")
print("  dataset_meta.json   — metadata")

Saved files:
  ideal_data.npy      — ideal (noiseless) expectation values, shape (2000, 5)
  noisy_data.npy      — noisy expectation values,             shape (2000, 5)
  theta_samples.npy   — parameter vectors,                    shape (2000, 8)
  dataset_meta.json   — metadata


## 7. Sanity Check

Verify the noise is meaningful — we expect mean absolute errors of ~0.01–0.10 per Pauli term.  
If errors are near zero, noise is too weak. If above 0.3, something is wrong.

In [12]:
errors = np.abs(ideal_data - noisy_data)

print(f"{'Pauli':<8} {'Ideal mean':>12} {'Noisy mean':>12} {'MAE':>10}")
print("-" * 46)
for j, label in enumerate(pauli_labels):
    print(f"{label:<8} {ideal_data[:, j].mean():>12.4f} "
          f"{noisy_data[:, j].mean():>12.4f} "
          f"{errors[:, j].mean():>10.4f}")

print(f"\nOverall MAE: {errors.mean():.4f}")
print(f"\n{'Good' if 0.005 < errors.mean() < 0.3 else 'WARNING: check noise level!'}: "
      f"noise level looks {'reasonable' if 0.005 < errors.mean() < 0.3 else 'unexpected'}.")

Pauli      Ideal mean   Noisy mean        MAE
----------------------------------------------
II             1.0000       1.0000     0.0000
IZ            -0.0102      -0.0096     0.0268
ZI             0.0095       0.0089     0.0213
ZZ            -0.0045      -0.0041     0.0453
XX            -0.0112      -0.0101     0.0449

Overall MAE: 0.0277

Good: noise level looks reasonable.
